# nb05b — RetinaNet R0 Evaluation & W&B Logging (resume)

**Purpose.** nb05 trained RetinaNet R0 successfully (1 epoch, `retinanet_R0.pth` saved) but crashed at the full-val eval cell with `AssertionError: targets should not be none when in training mode` — the model was left in `train()` mode after the training loop, and RetinaNet's forward asserts that targets are given when `self.training=True`.

**This notebook:**
1. Bootstraps from the `retinanet-1` dataset (checkpoint, train log, cached GT).
2. Loads `retinanet_R0.pth` into a fresh 9-class RetinaNet.
3. Fixes the eval bug (`model.eval()` before inference, and a permanent `detector.eval()` inside a local `run_detector` override — so this notebook is not vulnerable to the same bug even if `common.py` is unchanged).
4. Runs full-val COCO evaluation (2,846 frames, `iscrowd=1`).
5. Logs metrics + 3 prediction images + speed benchmark to W&B.
6. Saves `metrics_retinanet_R0.json` and `speed_retinanet_R0.json` to `/kaggle/working/` for the report.
7. Prints the completed §6 experiment-log row (nb05) that you'll paste into `draft_to_report.md`.

**No training in this notebook.** If you re-run it, it re-evaluates the same checkpoint — 12 minutes, no GPU-quota impact beyond the eval.

## 1. Bootstrap

Mounts the `retinanet-1` dataset and restores the paths + split + GT used in nb05. Same locked preprocessing: `letterbox-1280`, clip-disjoint split.

In [1]:
import sys, json, time, shutil, os
from pathlib import Path

import numpy as np
import pandas as pd
import cv2
import torch
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from torchvision.ops import box_iou
from torchvision.models.detection import retinanet_resnet50_fpn_v2
from torchvision.models.detection.retinanet import RetinaNetClassificationHead, RetinaNetRegressionHead
from torchvision.models.detection.anchor_utils import AnchorGenerator

# ---- Paths -----------------------------------------------------------------
COMP_DIR      = Path("/kaggle/input/competitions/multiple-object-video-detection-hw-2-ucu-ai-2026")
DATASET_DIR   = Path("/kaggle/input/datasets/suchok/my-dataset")
OUTPUTS_DIR   = Path("/kaggle/input/datasets/suchok/outputs")
RETINANET1_DIR = Path("/kaggle/input/datasets/suchok/retinanet-1")      # ← dataset from nb05 outputs
WORK_DIR      = Path("/kaggle/working")

assert COMP_DIR.exists(),       f"missing competition: {COMP_DIR}"
assert DATASET_DIR.exists(),    f"missing my-dataset: {DATASET_DIR}"
assert OUTPUTS_DIR.exists(),    f"missing outputs: {OUTPUTS_DIR}"
assert RETINANET1_DIR.exists(), f"missing retinanet-1 dataset: {RETINANET1_DIR}"

WORK_DIR.mkdir(parents=True, exist_ok=True)

# ---- common.py --------------------------------------------------------------
sys.path.insert(0, str(DATASET_DIR))
import common, importlib
importlib.reload(common)
from common import (
    set_seeds, SEED, CLASS_NAMES, NUM_CLASSES,
    load_splits, index_split, filter_by_clips,
    evaluate_full_val, coco_eval_summary,
    predict_with_strategy, benchmark_speed,
    save_metrics,
    IMAGENET_MEAN, IMAGENET_STD,
)
from torchvision.transforms import functional as TF

set_seeds(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("DEVICE:", DEVICE, "| torch:", torch.__version__)

# ---- Locked protocol --------------------------------------------------------
chosen = json.load(open(OUTPUTS_DIR / "chosen_preprocessing.json"))
STRAT  = chosen["strategy"]
SIZE   = int(chosen["size"])
print("Locked preprocessing:", STRAT, SIZE)

train_clips, val_clips = load_splits(OUTPUTS_DIR / "splits")
val_index = filter_by_clips(index_split(COMP_DIR / "val"), val_clips)
print(f"val frames: {len(val_index):,}")

# ---- Restore artifacts from retinanet-1 -------------------------------------
for name in ["retinanet_R0.pth", "gt_fullval_ignore.json", "train_log_retinanet_R0.json"]:
    src = RETINANET1_DIR / name
    dst = WORK_DIR / name
    if src.exists():
        shutil.copy(src, dst)
        print(f"restored: {name}  ({src.stat().st_size/1e6:.1f} MB)")
    else:
        print(f"⚠️  missing in retinanet-1: {name}")

gt_fullval_ignore = WORK_DIR / "gt_fullval_ignore.json"
ckpt_path         = WORK_DIR / "retinanet_R0.pth"
train_log_path    = WORK_DIR / "train_log_retinanet_R0.json"
assert gt_fullval_ignore.exists(), f"missing GT: {gt_fullval_ignore}"
assert ckpt_path.exists(),         f"missing checkpoint: {ckpt_path}"

# Print training summary from the saved log (proof that training completed)
if train_log_path.exists():
    with open(train_log_path) as f:
        train_log = json.load(f)
    print("\n--- nb05 training summary ---")
    for k, v in train_log.items():
        print(f"  {k}: {v}")

DEVICE: cuda | torch: 2.11.0+cu130
Locked preprocessing: letterbox 1280
val frames: 2,846
restored: retinanet_R0.pth  (145.6 MB)
restored: gt_fullval_ignore.json  (21.0 MB)
restored: train_log_retinanet_R0.json  (0.0 MB)

--- nb05 training summary ---
  epoch: 1
  mean_loss: 0.7960324883288589
  mean_loss_cls: 0.4163275994656007
  mean_loss_box: 0.37970488914096157
  wall_seconds: 15356.926017284393
  batch_size: 2
  lr: 0.005
  anchor_sizes: [32, 64, 128, 256, 512]
  anchor_aspect_ratios: [0.5, 1.0, 2.0]


## 2. Rebuild RetinaNet and load R0 weights

Same architecture as nb05: 9-class head, COCO-default anchors (32, 64, 128, 256, 512) with aspect ratios (0.5, 1.0, 2.0). Weights are loaded from `retinanet_R0.pth`. `norm_layer` uses `torch.nn.GroupNorm(32, C)` (the correct constructor — the nb05 bug was in `lambda x: group_norm(x, 32)`; that bug is **not** present here because we're rebuilding from the same fixed builder pattern used in nb05's Cell 15).

In [2]:
def build_retinanet_r0(anchor_sizes=(32, 64, 128, 256, 512),
                       anchor_aspect_ratios=(0.5, 1.0, 2.0),
                       score_thresh=0.05, nms_thresh=0.5):
    """RetinaNet with 9-class head and COCO-default anchors (matches R0)."""
    m = retinanet_resnet50_fpn_v2(
        weights=None, weights_backbone=None,   # weights come from checkpoint
        min_size=SIZE, max_size=SIZE,
        score_thresh=score_thresh, nms_thresh=nms_thresh,
        detections_per_img=500,
    )
    aspect_ratios_per_level = tuple(tuple(anchor_aspect_ratios) for _ in anchor_sizes)
    m.anchor_generator = AnchorGenerator(
        sizes=tuple((s,) for s in anchor_sizes),
        aspect_ratios=aspect_ratios_per_level,
    )
    num_anchors = m.anchor_generator.num_anchors_per_location()[0]

    in_channels = m.head.classification_head.cls_logits.in_channels
    m.head.classification_head = RetinaNetClassificationHead(
        in_channels=in_channels, num_anchors=num_anchors,
        num_classes=NUM_CLASSES + 1,
        norm_layer=lambda num_channels: torch.nn.GroupNorm(32, num_channels),
    )
    in_channels_reg = m.head.regression_head.bbox_reg.in_channels
    m.head.regression_head = RetinaNetRegressionHead(
        in_channels=in_channels_reg, num_anchors=num_anchors,
        norm_layer=lambda num_channels: torch.nn.GroupNorm(32, num_channels),
    )
    return m


model_r0 = build_retinanet_r0().to(DEVICE)
state = torch.load(ckpt_path, map_location=DEVICE)
missing, unexpected = model_r0.load_state_dict(state, strict=True)
print("loaded state_dict strictly — no missing/unexpected keys")
model_r0.eval()                    # ← THE FIX: eval mode, not train
n_params = sum(p.numel() for p in model_r0.parameters() if p.requires_grad)
print(f"trainable params: {n_params/1e6:.2f}M | mode: {'train' if model_r0.training else 'eval'}")

loaded state_dict strictly — no missing/unexpected keys
trainable params: 36.32M | mode: eval


## 3. W&B login

Log to the same `visdrone-det` project. This run is named `retinanet_R0_eval` so it does not overwrite the training run `retinanet_R0` from nb05.

In [3]:
from kaggle_secrets import UserSecretsClient
os.environ["WANDB_API_KEY"] = UserSecretsClient().get_secret("WANDB_API_KEY")

import wandb
wandb.login(key=os.environ["WANDB_API_KEY"], relogin=True)

WANDB_PROJECT = "visdrone-det"

run = wandb.init(
    project=WANDB_PROJECT,
    name="retinanet_R0_eval",
    config={
        "model": "retinanet_resnet50_fpn_v2",
        "variant": "R0_baseline_eval",
        "strategy": STRAT, "size": SIZE,
        "anchor_sizes": [32, 64, 128, 256, 512],
        "anchor_aspect_ratios": [0.5, 1.0, 2.0],
        "checkpoint": "retinanet_R0.pth (from nb05)",
        "seed": SEED,
    },
    reinit=True,
)

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: suchok-pn (suchok-pn-ukrainian-catholic-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.
wandb: Tracking run with wandb version 0.28.1
wandb: Run data is saved locally in /kaggle/working/wandb/run-20261107_125658-gobhax6x
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run retinanet_R0_eval
wandb: ⭐️ View project at https://wandb.ai/suchok-pn-ukrainian-catholic-university/visdrone-det


## 4. Local `run_detector` override — the actual fix

`common.run_detector` calls `detector([tensor])` without calling `.eval()` first. It works for Faster R-CNN by luck (no assertion) but **RetinaNet crashes** in training mode when `targets=None`.

The cleanest fix is inside `run_detector`, but since `common.py` is a read-only Kaggle input, we override it **locally** in this notebook. This has the side benefit of being a safe, reusable pattern for nb06/07/08/09/11 — copy this cell into every future notebook.

**The override differs from `common.run_detector` by exactly one line: `detector.eval()` before the forward pass.**

In [4]:
@torch.no_grad()
def run_detector_eval(detector, image_rgb, score_thr=0.001, device=None):
    """Same as common.run_detector, but forces eval() before inference.

    Fixes: RetinaNet raises AssertionError when targets=None in train() mode.
    Faster R-CNN silently tolerated this, RetinaNet does not.
    """
    if device is None:
        device = next(detector.parameters()).device
    detector.eval()                                    # ← THE FIX
    tensor = TF.to_tensor(image_rgb)
    tensor = TF.normalize(tensor, mean=IMAGENET_MEAN, std=IMAGENET_STD)
    out = detector([tensor.to(device)])[0]
    keep = out["scores"] >= score_thr
    return (out["boxes"][keep].cpu().numpy(),
            out["labels"][keep].cpu().numpy(),
            out["scores"][keep].cpu().numpy())


def predict_with_strategy_eval(detector, image_path, strategy, size,
                               score_thr=0.001, device=None):
    """Full pipeline: read -> strategy -> detect -> map back, using safe eval."""
    import cv2
    from common import apply_strategy, boxes_to_original

    img = cv2.cvtColor(cv2.imread(image_path), cv2.COLOR_BGR2RGB)
    h0, w0 = img.shape[:2]
    inp, meta = apply_strategy(img, strategy, size, seed_key=image_path)

    t0 = time.time()
    boxes, labels, scores = run_detector_eval(detector, inp,
                                              score_thr=score_thr, device=device)
    dt = time.time() - t0

    boxes_orig = boxes_to_original(boxes, meta, strategy, w0, h0)
    return boxes_orig, labels, scores, dt, meta


# Quick smoke test: one val image, expect non-empty output
_row = val_index.iloc[0]
_boxes, _labels, _scores, _dt, _ = predict_with_strategy_eval(
    model_r0, _row["image_path"], STRAT, SIZE, score_thr=0.3, device=DEVICE
)
print(f"smoke eval on 1 image: {len(_boxes)} detections | {_dt*1000:.0f} ms")
assert len(_boxes) > 0, "no detections at score_thr=0.3 — the model is broken"

smoke eval on 1 image: 32 detections | 6223 ms


## 5. Full-val evaluation

Runs `predict_with_strategy_eval` over all 2,846 val frames, writes a COCO-format JSON, then calls `coco_eval_summary` (which uses `pycocotools` with `iscrowd=1` GT).

**Expected runtime:** ~10–12 minutes on T4. **Expected output:** a `val_preds_retinanet_R0.json` (~5–15 MB) and a metrics dict.

In [5]:
import contextlib, io
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval


def predictions_to_coco_eval(df_index, detector, strategy, size, out_json,
                              score_thr=0.001, device=None):
    """Same as common.predictions_to_coco but uses predict_with_strategy_eval."""
    preds = []
    id_map = {row["image_path"]: i for i, (_, row) in enumerate(
        df_index.iterrows(), start=1)}
    valid_cat_ids = set(range(1, NUM_CLASSES + 1))

    for _, row in tqdm(df_index.iterrows(), total=len(df_index),
                       desc=f"pred {strategy}-{size} (eval-safe)"):
        boxes, labels, scores, _, _ = predict_with_strategy_eval(
            detector, row["image_path"], strategy, size,
            score_thr=score_thr, device=device)
        img_id = id_map[row["image_path"]]
        for (x1, y1, x2, y2), l, s in zip(boxes, labels, scores):
            if int(l) not in valid_cat_ids:
                continue
            preds.append({
                "image_id": int(img_id),
                "category_id": int(l),
                "bbox": [float(x1), float(y1), float(x2 - x1), float(y2 - y1)],
                "score": float(s),
            })

    with open(out_json, "w") as f:
        json.dump(preds, f)
    print(f"wrote {len(preds):,} predictions -> {out_json}")


def coco_eval_summary_safe(gt_json, dt_json):
    """Same as common.coco_eval_summary but returns metrics directly."""
    with contextlib.redirect_stdout(io.StringIO()):
        coco_gt = COCO(gt_json)
        if os.path.getsize(dt_json) == 0 or json.load(open(dt_json)) == []:
            return {k: 0.0 for k in ["mAP", "AP50", "AP75",
                                     "AP_small", "AP_medium", "AP_large"]}
        coco_dt = coco_gt.loadRes(dt_json)
        e = COCOeval(coco_gt, coco_dt, "bbox")
        e.evaluate(); e.accumulate(); e.summarize()
    return {
        "mAP":       float(e.stats[0]),
        "AP50":      float(e.stats[1]),
        "AP75":      float(e.stats[2]),
        "AP_small":  float(e.stats[3]),
        "AP_medium": float(e.stats[4]),
        "AP_large":  float(e.stats[5]),
    }


torch.cuda.empty_cache()
model_r0.eval()

preds_path_r0 = WORK_DIR / "val_preds_retinanet_R0.json"
t0 = time.time()
predictions_to_coco_eval(
    val_index, model_r0, STRAT, SIZE, str(preds_path_r0),
    score_thr=0.001, device=DEVICE,
)
print(f"inference time: {(time.time()-t0)/60:.2f} min")

t0 = time.time()
metrics_r0 = coco_eval_summary_safe(str(gt_fullval_ignore), str(preds_path_r0))
print(f"COCOeval time: {time.time()-t0:.1f} s")

print("=" * 60)
print("R0 (baseline) — full val:")
for k, v in metrics_r0.items():
    print(f"  {k:10s}: {v:.4f}")

save_metrics(metrics_r0, WORK_DIR / "metrics_retinanet_R0.json")
wandb.log({f"val/{k}": v for k, v in metrics_r0.items()})

pred letterbox-1280 (eval-safe):   0%|          | 0/2846 [00:00<?, ?it/s]

wrote 1,414,367 predictions -> /kaggle/working/val_preds_retinanet_R0.json
inference time: 16.48 min
COCOeval time: 176.3 s
R0 (baseline) — full val:
  mAP       : 0.0410
  AP50      : 0.0941
  AP75      : 0.0310
  AP_small  : 0.0148
  AP_medium : 0.0352
  AP_large  : 0.1053


## 6. Speed benchmark

`benchmark_speed` from `common.py` calls `predict_with_strategy`, which uses the **buggy** `common.run_detector`. Rather than call it, we time 50 image forwards directly with `predict_with_strategy_eval`. The result is the FPS number for the report table.

**Note:** the report's FPS row for R0 is inference-only (no NMS suppression counting). This matches nb04c's FPS reporting convention.

In [6]:
def benchmark_speed_eval(detector, df_index, strategy, size, n=50,
                          device=None, score_thr=0.5, warmup=5):
    """Same as common.benchmark_speed but uses the eval-safe pipeline."""
    detector.eval()
    if device is None:
        device = next(detector.parameters()).device

    # warmup
    for i in range(warmup):
        row = df_index.iloc[i % len(df_index)]
        _ = predict_with_strategy_eval(detector, row["image_path"], strategy, size,
                                        score_thr=score_thr, device=device)
    if device == "cuda":
        torch.cuda.synchronize()

    times = []
    for i in range(n):
        row = df_index.iloc[(warmup + i) % len(df_index)]
        if device == "cuda":
            torch.cuda.synchronize()
        t0 = time.time()
        _ = predict_with_strategy_eval(detector, row["image_path"], strategy, size,
                                        score_thr=score_thr, device=device)
        if device == "cuda":
            torch.cuda.synchronize()
        times.append(time.time() - t0)

    mean_s = float(np.mean(times))
    return {
        "device": str(device),
        "gpu_ms_per_image": mean_s * 1000,
        "gpu_fps":          1.0 / mean_s,
        "n_images":         n,
        "score_thr":        score_thr,
    }


speed_r0 = benchmark_speed_eval(model_r0, val_index, STRAT, SIZE,
                                 n=50, device=DEVICE, score_thr=0.5, warmup=5)
print("speed:", json.dumps(speed_r0, indent=2))

with open(WORK_DIR / "speed_retinanet_R0.json", "w") as f:
    json.dump(speed_r0, f, indent=2)
wandb.log({"speed/gpu_fps": speed_r0["gpu_fps"],
           "speed/gpu_ms":  speed_r0["gpu_ms_per_image"]})

speed: {
  "device": "cuda",
  "gpu_ms_per_image": 294.65219497680664,
  "gpu_fps": 3.3938318364766107,
  "n_images": 50,
  "score_thr": 0.5
}


## 7. Qualitative check — log 3 val predictions to W&B

Picks 3 diverse frames (first / middle / last val index), draws the model's predictions at `score_thr=0.3`, and uploads them as `wandb.Image` with boxes. Satisfies the bonus-rubric requirement *"validation predictions with bounding boxes"* for R0.

In [7]:
def _draw_and_convert_eval(image_path, preds_json, image_id, thr=0.3):
    """Draw prediction boxes on an image and return a wandb.Image."""
    img = cv2.cvtColor(cv2.imread(str(image_path)), cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    with open(preds_json) as f:
        preds = [p for p in json.load(f)
                 if p["image_id"] == image_id and p["score"] >= thr]
    boxes_data = []
    for p in preds:
        x, y, bw, bh = p["bbox"]
        boxes_data.append({
            "position": {"minX": x / w, "maxX": (x + bw) / w,
                         "minY": y / h, "maxY": (y + bh) / h},
            "class_id": p["category_id"] - 1,
            "box_caption": f"{CLASS_NAMES[p['category_id']-1]} {p['score']:.2f}",
            "scores": {"score": p["score"]},
        })
    return wandb.Image(img, boxes={"predictions": {"box_data": boxes_data}})


sample_ids = [1, len(val_index) // 2, len(val_index)]
sample_imgs = []
for sid in sample_ids:
    row = val_index.iloc[sid - 1]
    sample_imgs.append(
        _draw_and_convert_eval(row["image_path"], preds_path_r0, sid, thr=0.3)
    )

wandb.log({"val/predictions": sample_imgs})
print(f"logged {len(sample_imgs)} prediction images")

logged 3 prediction images


## 8. Session summary + report row

Prints the completed §6 experiment-log row for `draft_to_report.md`. Copy this into the report table after the run.

In [8]:
print("=" * 70)
print("NOTEBOOK COMPLETE — copy this row into draft_to_report.md §6")
print("=" * 70)
print()
print(f"| nb05 | RetinaNet R0 | COCO default anchors, focal γ=2, 1 epoch | "
      f"{metrics_r0['mAP']:.4f} | "
      f"{metrics_r0['AP50']:.4f} | "
      f"{metrics_r0['AP_small']:.4f} | "
      f"{speed_r0['gpu_fps']:.2f} FPS | "
      f"reference |")
print()
print("=" * 70)
print("Artifacts in /kaggle/working/:")
for p in sorted(WORK_DIR.glob("*retinanet_R0*")):
    print(f"  {p.name}  ({p.stat().st_size/1e6:.1f} MB)")
print()
print("Next: nb06 — RetinaNet R1 (data-grounded anchors), warm-start from retinanet_R0.pth")
print("=" * 70)

run.finish()
print("W&B run finished:", run.url)

wandb: updating run metadata


NOTEBOOK COMPLETE — copy this row into draft_to_report.md §6

| nb05 | RetinaNet R0 | COCO default anchors, focal γ=2, 1 epoch | 0.0410 | 0.0941 | 0.0148 | 3.39 FPS | reference |

Artifacts in /kaggle/working/:
  metrics_retinanet_R0.json  (0.0 MB)
  retinanet_R0.pth  (145.6 MB)
  speed_retinanet_R0.json  (0.0 MB)
  train_log_retinanet_R0.json  (0.0 MB)
  val_preds_retinanet_R0.json  (210.0 MB)

Next: nb06 — RetinaNet R1 (data-grounded anchors), warm-start from retinanet_R0.pth


wandb: uploading history steps 2-2, summary, console lines 19-36
wandb: 
wandb: Run history:
wandb: speed/gpu_fps ▁
wandb:  speed/gpu_ms ▁
wandb:      val/AP50 ▁
wandb:      val/AP75 ▁
wandb:  val/AP_large ▁
wandb: val/AP_medium ▁
wandb:  val/AP_small ▁
wandb:       val/mAP ▁
wandb: 
wandb: Run summary:
wandb: speed/gpu_fps 3.39383
wandb:  speed/gpu_ms 294.65219
wandb:      val/AP50 0.09412
wandb:      val/AP75 0.03104
wandb:  val/AP_large 0.1053
wandb: val/AP_medium 0.03523
wandb:  val/AP_small 0.0148
wandb:       val/mAP 0.04104
wandb: 
wandb: 🚀 View run retinanet_R0_eval at: https://wandb.ai/suchok-pn-ukrainian-catholic-university/visdrone-det/runs/gobhax6x
wandb: ⭐️ View project at: https://wandb.ai/suchok-pn-ukrainian-catholic-university/visdrone-det
wandb: Synced 5 W&B file(s), 6 media file(s), 0 artifact file(s) and 0 other file(s)
wandb: Find logs at: ./wandb/run-20261107_125658-gobhax6x/logs


W&B run finished: https://wandb.ai/suchok-pn-ukrainian-catholic-university/visdrone-det/runs/gobhax6x
